In [1]:
# ============================================================
# Task 1: Data Cleaning & Preprocessing — Titanic Dataset
# ElevateLabs AI/ML Internship
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# ── Set plot style ───────────────────────────────────────────
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


In [2]:
# ============================================================
# STEP 1 — Load & Explore the Dataset
# ============================================================
print("=" * 60)
print("STEP 1: Loading & Exploring the Dataset")
print("=" * 60)

df = pd.read_csv("Titanic-Dataset.csv")

print("\n📌 Shape:", df.shape)
print("\n📌 First 5 rows:")
print(df.head())

print("\n📌 Data Types:")
print(df.dtypes)

print("\n📌 Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})[missing > 0])

print("\n📌 Basic Statistics:")
print(df.describe())

STEP 1: Loading & Exploring the Dataset

📌 Shape: (891, 12)

📌 First 5 rows:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.

In [3]:
# ============================================================
# STEP 2 — Handle Missing Values
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Handling Missing Values")
print("=" * 60)

# Age  → fill with median (skewed distribution, robust to outliers)
age_median = df["Age"].median()
df["Age"] = df["Age"].fillna(age_median)
print(f"✅ 'Age' filled with median: {age_median}")

# Embarked → fill with mode (only 2 missing)
embarked_mode = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(embarked_mode)
print(f"✅ 'Embarked' filled with mode: {embarked_mode}")

# Cabin → too many missing (~77%), drop it
df.drop(columns=["Cabin"], inplace=True)
print("✅ 'Cabin' column dropped (>77% missing)")

print("\n📌 Missing values after treatment:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("No missing values remaining ✔" if df.isnull().sum().sum() == 0 else "")


STEP 2: Handling Missing Values
✅ 'Age' filled with median: 28.0
✅ 'Embarked' filled with mode: S
✅ 'Cabin' column dropped (>77% missing)

📌 Missing values after treatment:
Series([], dtype: int64)
No missing values remaining ✔


In [4]:
# ============================================================
# STEP 3 — Drop Irrelevant Columns
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Dropping Irrelevant Columns")
print("=" * 60)

# PassengerId, Name, Ticket — not useful for ML
df.drop(columns=["PassengerId", "Name", "Ticket"], inplace=True)
print("✅ Dropped: PassengerId, Name, Ticket")
print("📌 Remaining columns:", df.columns.tolist())



STEP 3: Dropping Irrelevant Columns
✅ Dropped: PassengerId, Name, Ticket
📌 Remaining columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


In [5]:
# ============================================================
# STEP 4 — Encode Categorical Features
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Encoding Categorical Features")
print("=" * 60)

# Label Encoding — Sex (binary: male=1, female=0)
df["Sex"] = df["Sex"].map({"male": 1, "female": 0})
print("✅ 'Sex' label-encoded → male=1, female=0")

# One-Hot Encoding — Embarked (3 categories: S, C, Q)
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)
print("✅ 'Embarked' one-hot encoded (drop_first=True to avoid dummy trap)")
print("📌 Columns after encoding:", df.columns.tolist())


STEP 4: Encoding Categorical Features
✅ 'Sex' label-encoded → male=1, female=0
✅ 'Embarked' one-hot encoded (drop_first=True to avoid dummy trap)
📌 Columns after encoding: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']


In [6]:
# ============================================================
# STEP 5 — Visualize Outliers with Boxplots
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Visualizing Outliers")
print("=" * 60)

numerical_cols = ["Age", "Fare", "SibSp", "Parch"]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, numerical_cols):
    sns.boxplot(y=df[col], ax=ax, color="skyblue")
    ax.set_title(f"Boxplot: {col}")
plt.suptitle("Outlier Visualization — Before Removal", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("boxplots_before.png", dpi=150)
plt.close()
print("✅ Saved: boxplots_before.png")


STEP 5: Visualizing Outliers
✅ Saved: boxplots_before.png


In [7]:
# ============================================================
# STEP 6 — Remove Outliers using IQR Method
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: Removing Outliers (IQR Method)")
print("=" * 60)

def remove_outliers_iqr(dataframe, columns):
    df_clean = dataframe.copy()
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        before = len(df_clean)
        df_clean = df_clean[(df_clean[col] >= lower) & (df_clean[col] <= upper)]
        removed = before - len(df_clean)
        print(f"  '{col}': IQR=[{lower:.2f}, {upper:.2f}] → {removed} rows removed")
    return df_clean

df = remove_outliers_iqr(df, ["Fare", "SibSp", "Parch"])
print(f"\n📌 Shape after outlier removal: {df.shape}")

# Boxplots after removal
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, numerical_cols):
    if col in df.columns:
        sns.boxplot(y=df[col], ax=ax, color="lightgreen")
        ax.set_title(f"Boxplot: {col}")
plt.suptitle("Outlier Visualization — After Removal", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("boxplots_after.png", dpi=150)
plt.close()
print("✅ Saved: boxplots_after.png")


STEP 6: Removing Outliers (IQR Method)
  'Fare': IQR=[-26.72, 65.63] → 116 rows removed
  'SibSp': IQR=[-1.50, 2.50] → 36 rows removed
  'Parch': IQR=[0.00, 0.00] → 132 rows removed

📌 Shape after outlier removal: (607, 9)
✅ Saved: boxplots_after.png


In [8]:
# ============================================================
# STEP 7 — Feature Scaling
# ============================================================
print("\n" + "=" * 60)
print("STEP 7: Feature Scaling")
print("=" * 60)

scale_cols = ["Age", "Fare"]

# StandardScaler (Z-score normalization: mean=0, std=1)
scaler = StandardScaler()
df_standardized = df.copy()
df_standardized[scale_cols] = scaler.fit_transform(df[scale_cols])

# MinMaxScaler (Normalization: range [0, 1])
minmax = MinMaxScaler()
df_normalized = df.copy()
df_normalized[scale_cols] = minmax.fit_transform(df[scale_cols])

print("✅ StandardScaler applied to Age and Fare")
print("✅ MinMaxScaler applied to Age and Fare")

print("\n📌 Sample — Standardized values:")
print(df_standardized[scale_cols].describe().round(3))

# Save the standardized version as final output
df_final = df_standardized.copy()
df_final.to_csv("titanic_cleaned.csv", index=False)
print("\n✅ Final cleaned dataset saved: titanic_cleaned.csv")


STEP 7: Feature Scaling
✅ StandardScaler applied to Age and Fare
✅ MinMaxScaler applied to Age and Fare

📌 Sample — Standardized values:
           Age     Fare
count  607.000  607.000
mean     0.000    0.000
std      1.001    1.001
min     -2.343   -1.208
25%     -0.536   -0.591
50%     -0.265   -0.506
75%      0.299    0.121
max      4.432    3.645

✅ Final cleaned dataset saved: titanic_cleaned.csv


In [9]:
# ============================================================
# STEP 8 — Final Summary
# ============================================================
print("\n" + "=" * 60)
print("STEP 8: Final Dataset Summary")
print("=" * 60)
print("📌 Shape:", df_final.shape)
print("📌 Columns:", df_final.columns.tolist())
print("📌 Missing Values:", df_final.isnull().sum().sum())
print("\n📌 First 5 rows of cleaned data:")
print(df_final.head())

# Correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df_final.corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap — Cleaned Dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.close()
print("✅ Saved: correlation_heatmap.png")

print("\n🎉 Task 1 Complete! All outputs saved.")



STEP 8: Final Dataset Summary
📌 Shape: (607, 9)
📌 Columns: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_Q', 'Embarked_S']
📌 Missing Values: 0

📌 First 5 rows of cleaned data:
   Survived  Pclass  Sex       Age  SibSp  Parch      Fare  Embarked_Q  \
0         0       3    1 -0.807162      1      0 -0.632732       False   
2         1       3    0 -0.445844      0      0 -0.579187       False   
3         1       1    0  0.367122      1      0  3.004335       False   
4         0       3    1  0.367122      0      0 -0.569271       False   
5         0       3    1 -0.265185      0      0 -0.536883        True   

   Embarked_S  
0        True  
2        True  
3        True  
4        True  
5       False  
✅ Saved: correlation_heatmap.png

🎉 Task 1 Complete! All outputs saved.
